# 第4章 探索性分析与可视化 —— 配套代码

对应正文 `book/part1/04-eda-visualization.md`。依赖内置数据，**离线可跑**。

**运行顺序**：先执行 Cell 1（环境初始化+数据加载），再按顺序执行后续 cell。

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# 环境初始化：导入、字体、数据
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import jarque_bera, norm, gaussian_kde
from statsmodels.tsa.stattools import acf as sm_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from matplotlib.patches import Patch

from fds import load_sample_prices, daily_returns, set_chinese_font, max_drawdown, annualized_volatility

set_chinese_font()

prices = load_sample_prices()
rets = daily_returns(prices)

print(f"价格数据形状：{prices.shape}，时间范围：{prices.index[0].date()} ~ {prices.index[-1].date()}")
print(f"收益率数据形状：{rets.shape}")
print(f"股票列表：{list(rets.columns)}")

## 4.2 描述统计：四个矩与分位数汇总表

In [ ]:
# 描述性统计汇总表
TRADING_DAYS = 252

summary = pd.DataFrame({
    '日均收益(%)': rets.mean() * 100,
    '年化波动率(%)': rets.std() * np.sqrt(TRADING_DAYS) * 100,
    '偏度': rets.skew(),
    '超额峰度': rets.kurtosis(),
    '历史最大日跌幅(%)': rets.min() * 100,
    '历史最大日涨幅(%)': rets.max() * 100,
    '1%分位数(%)': rets.quantile(0.01) * 100,
    '99%分位数(%)': rets.quantile(0.99) * 100,
}).T

print('四只股票描述性统计汇总：')
summary.round(3)

## 4.3 分布分析：直方图 + KDE + 正态曲线对比

四张子图并列，每张叠加：直方图、KDE 曲线、正态拟合曲线。

In [ ]:
# 直方图 + KDE + 正态曲线
set_chinese_font()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('四只股票日收益率分布（直方图 + KDE vs 正态）', fontsize=14)

for ax, col in zip(axes.ravel(), rets.columns):
    data = rets[col].dropna()
    mu, sigma = data.mean(), data.std()
    ax.hist(data, bins=60, density=True, alpha=0.35, color='steelblue', label='直方图')
    x_range = np.linspace(data.min(), data.max(), 300)
    kde = gaussian_kde(data)
    ax.plot(x_range, kde(x_range), color='steelblue', lw=2, label='KDE')
    ax.plot(x_range, norm.pdf(x_range, mu, sigma), 'r--', lw=2, label='正态拟合')
    ax.set_title(f'{col}  超额峰度={data.kurtosis():.2f}')
    ax.set_xlabel('日收益率')
    ax.set_ylabel('概率密度')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4.3 分布诊断：四张 QQ 图（厚尾可视化）

两端向外翻翘（sigmoid 形）是厚尾的典型特征。

In [ ]:
# 四张 QQ 图
set_chinese_font()

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
fig.suptitle('四只股票正态 QQ 图（两端翘起 = 厚尾）', fontsize=14)

for ax, col in zip(axes.ravel(), rets.columns):
    data = rets[col].dropna()
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm')
    ax.scatter(osm, osr, s=8, alpha=0.5, color='steelblue', label='样本分位数')
    xline = np.array([osm[0], osm[-1]])
    ax.plot(xline, slope * xline + intercept, 'r-', lw=1.5, label='正态参考线')
    ax.set_title(f'{col}  超额峰度={data.kurtosis():.2f}')
    ax.set_xlabel('正态理论分位数')
    ax.set_ylabel('样本分位数')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4.5 正态性检验：Jarque-Bera

p 值极小则强烈拒绝正态性假设。

In [ ]:
# Jarque-Bera 正态性检验
jb_results = []
for col in rets.columns:
    data = rets[col].dropna()
    stat, p = jarque_bera(data)
    jb_results.append({
        '股票': col,
        '样本量': len(data),
        '偏度': round(data.skew(), 4),
        '超额峰度': round(data.kurtosis(), 4),
        'JB统计量': round(stat, 1),
        'p值': f'{p:.2e}',
        '结论': '拒绝正态' if p < 0.05 else '不拒绝正态'
    })

jb_df = pd.DataFrame(jb_results).set_index('股票')
print('Jarque-Bera 正态性检验结果（原假设：服从正态分布）')
jb_df

## 4.4 风格化事实①：波动率聚集 —— ACF 对比图

收益率 ACF 接近 0，绝对值 ACF 显著且缓慢衰减（波动率持续聚集）。

In [ ]:
# ACF 对比图：收益率 vs 收益率绝对值
set_chinese_font()

NLAGS = 30
conf_bound = 1.96 / np.sqrt(len(rets))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('各股票 ACF 对比：日收益率（蓝）vs 绝对值（橙）', fontsize=13)

for ax, col in zip(axes.ravel(), rets.columns):
    r = rets[col].dropna().values
    lags = np.arange(1, NLAGS + 1)
    acf_ret = sm_acf(r, nlags=NLAGS, fft=True)[1:]
    acf_abs = sm_acf(np.abs(r), nlags=NLAGS, fft=True)[1:]
    width = 0.35
    ax.bar(lags - width/2, acf_ret, width=width, color='steelblue', alpha=0.8, label='收益率')
    ax.bar(lags + width/2, acf_abs, width=width, color='darkorange', alpha=0.8, label='|收益率|')
    ax.axhline(conf_bound, color='gray', ls='--', lw=1)
    ax.axhline(-conf_bound, color='gray', ls='--', lw=1)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(col)
    ax.set_xlabel('滞后阶数')
    ax.set_ylabel('自相关系数')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4.6 Ljung-Box 检验：收益率 vs 绝对值的自相关显著性

In [ ]:
# Ljung-Box 检验
lags_to_test = [5, 10, 20]
lb_rows = []

for col in rets.columns:
    r = rets[col].dropna()
    lb_ret = acorr_ljungbox(r, lags=lags_to_test, return_df=True)
    lb_abs = acorr_ljungbox(r.abs(), lags=lags_to_test, return_df=True)
    for lag in lags_to_test:
        lb_rows.append({
            '股票': col, '序列': '收益率', '滞后阶数': lag,
            'LB统计量': round(lb_ret.loc[lag, 'lb_stat'], 2),
            'p值': f"{lb_ret.loc[lag, 'lb_pvalue']:.3f}",
        })
        lb_rows.append({
            '股票': col, '序列': '|收益率|', '滞后阶数': lag,
            'LB统计量': round(lb_abs.loc[lag, 'lb_stat'], 2),
            'p值': f"{lb_abs.loc[lag, 'lb_pvalue']:.3e}",
        })

lb_df = pd.DataFrame(lb_rows)
print('Ljung-Box 检验（原假设：无自相关）')
lb_df

## 4.4 风格化事实②：杠杆效应验证

In [ ]:
# 杠杆效应简单验证
leverage_rows = []

for col in rets.columns:
    r = rets[col].dropna()
    q20 = r.quantile(0.20)
    q80 = r.quantile(0.80)
    mask_down = r < q20
    mask_up   = r > q80
    next_day_abs = r.shift(-1).abs()
    vol_after_down = next_day_abs[mask_down].mean()
    vol_after_up   = next_day_abs[mask_up].mean()
    leverage_rows.append({
        '股票': col,
        '大跌后次日均|收益|(%)': round(vol_after_down * 100, 4),
        '大涨后次日均|收益|(%)': round(vol_after_up * 100, 4),
        '杠杆效应比值': round(vol_after_down / vol_after_up, 3),
    })

lev_df = pd.DataFrame(leverage_rows).set_index('股票')
print('杠杆效应验证（大跌后波动率 > 大涨后波动率 即为正杠杆效应）')
lev_df

## 4.4 风格化事实③：聚合高斯性

In [ ]:
# 聚合高斯性
set_chinese_font()

freq_map = {'日度': 'D', '周度': 'W', '月度': 'ME'}
agg_kurt = {}

for label, freq in freq_map.items():
    if freq == 'D':
        agg_rets = rets
    else:
        agg_rets = prices.resample(freq).last().pct_change().dropna()
    agg_kurt[label] = agg_rets.kurtosis()

kurt_df = pd.DataFrame(agg_kurt).T
print('各频率下超额峰度（正态=0，越大越厚尾）')
print(kurt_df.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
kurt_df.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='k', linewidth=0.5)
ax.axhline(0, color='black', lw=1)
ax.set_title('聚合高斯性：频率越低超额峰度越趋近 0')
ax.set_xlabel('收益率频率')
ax.set_ylabel('超额峰度')
ax.legend(title='股票', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4.7 相关性分析：Pearson vs Spearman 热力图

In [ ]:
# Pearson vs Spearman 热力图
set_chinese_font()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('相关系数矩阵热力图', fontsize=13)

for ax, method in zip(axes, ['pearson', 'spearman']):
    corr = rets.corr(method=method)
    sns.heatmap(
        corr, annot=True, cmap='coolwarm',
        vmin=-1, vmax=1, ax=ax,
        fmt='.2f', linewidths=0.5,
        annot_kws={'size': 10}
    )
    label_cn = 'Pearson 线性相关' if method == 'pearson' else 'Spearman 秩相关'
    ax.set_title(label_cn)

plt.tight_layout()
plt.show()

## 4.7 滚动相关系数（相关性时变）

In [ ]:
# 60 日滚动相关系数
set_chinese_font()

pairs = [('BANK', 'LIQUOR'), ('BANK', 'TECH'), ('BANK', 'UTILITY'), ('LIQUOR', 'TECH')]

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
fig.suptitle('60 日滚动 Pearson 相关系数（相关性时变性）', fontsize=13)

for ax, (s1, s2) in zip(axes.ravel(), pairs):
    roll = rets[s1].rolling(60).corr(rets[s2])
    roll.plot(ax=ax, color='steelblue', lw=1.2)
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    mean_val = roll.mean()
    ax.axhline(mean_val, color='red', lw=1, ls='-.',
               label=f'均值={mean_val:.2f}')
    ax.set_title(f'{s1} vs {s2}')
    ax.set_ylabel('相关系数')
    ax.legend(fontsize=8)
    ax.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

## 4.8 金融专用可视化①：累计净值 + 回撤双子图

In [ ]:
# 累计净值 + 回撤双子图
set_chinese_font()

nav = (1 + rets).cumprod()

def drawdown_series(ret_series):
    cum = (1 + ret_series).cumprod()
    running_max = cum.cummax()
    return cum / running_max - 1

dd = pd.DataFrame({col: drawdown_series(rets[col]) for col in rets.columns})

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                 gridspec_kw={'height_ratios': [2, 1]},
                                 sharex=True)

for col in nav.columns:
    ax1.plot(nav.index, nav[col], lw=1.5, label=col)
ax1.set_title('四只股票累计净值（初始值=1）')
ax1.set_ylabel('净值')
ax1.legend()
ax1.axhline(1, color='gray', lw=0.8, ls='--')

for col in dd.columns:
    ax2.fill_between(dd.index, dd[col], 0, alpha=0.4, label=col)
ax2.set_title('回撤曲线')
ax2.set_ylabel('回撤幅度')
ax2.set_xlabel('日期')
ax2.legend(ncol=2)

plt.tight_layout()
plt.show()

mdd_table = pd.DataFrame({
    '最大回撤(%)': {col: round(max_drawdown(rets[col]) * 100, 2) for col in rets.columns}
})
print('\n最大回撤汇总：')
mdd_table

## 4.8 金融专用可视化②：按年收益分布箱线图

In [ ]:
# 按年收益分布箱线图
set_chinese_font()

rets_copy = rets.copy()
rets_copy['year'] = rets_copy.index.year
years = sorted(rets_copy['year'].unique())

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True)
fig.suptitle('各股票按年日收益率分布（箱线图）', fontsize=13)

for ax, col in zip(axes.ravel(), rets.columns):
    data_by_year = [rets_copy.loc[rets_copy['year'] == y, col].dropna().values for y in years]
    bp = ax.boxplot(data_by_year, labels=years, patch_artist=True,
                    flierprops=dict(marker='.', markersize=2, alpha=0.4))
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.6)
    ax.axhline(0, color='red', lw=0.8, ls='--')
    ax.set_title(col)
    ax.set_xlabel('年份')
    ax.set_ylabel('日收益率')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4.9 中文绘图规范演示

A股红涨绿跌配色 + 多图共享坐标轴 + 完整标注。

In [ ]:
# A股配色规范 + 共享坐标轴演示
set_chinese_font()

RISE_COLOR = '#e03c31'
FALL_COLOR = '#00a651'

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True)
fig.suptitle('四只股票日收益率时序图（A股红涨绿跌配色，共享坐标轴）', fontsize=13)

for ax, col in zip(axes.ravel(), rets.columns):
    r = rets[col]
    colors = [RISE_COLOR if v >= 0 else FALL_COLOR for v in r]
    ax.bar(r.index, r.values, color=colors, width=1, alpha=0.7)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(col)
    ax.set_ylabel('日收益率')

axes[1, 0].set_xlabel('日期')
axes[1, 1].set_xlabel('日期')

legend_elements = [
    Patch(facecolor=RISE_COLOR, label='上涨（红色，A股习惯）'),
    Patch(facecolor=FALL_COLOR, label='下跌（绿色，A股习惯）')
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=2, bbox_to_anchor=(0.5, -0.02), fontsize=10)

plt.tight_layout()
plt.show()

---
## 习题参考解答

以下为第4章习题的参考解答代码，可直接运行验证。

In [ ]:
# 习题解答 1：四只股票描述性统计汇总表
hw1 = pd.DataFrame({
    '日均收益(%)': (rets.mean() * 100).round(4),
    '年化波动率(%)': (annualized_volatility(rets) * 100).round(2),
    '偏度': rets.skew().round(4),
    '超额峰度': rets.kurtosis().round(4),
    '历史最大日跌幅(%)': (rets.min() * 100).round(4),
})
print('习题解答 1：')
print(hw1)
best = hw1['超额峰度'].idxmax()
print(f'\n超额峰度最大的股票：{best}')
print('含义：尾部最厚，极端涨跌频率最高，正态VaR最低估其尾部风险。')

In [ ]:
# 习题解答 2：厚尾程度排名
kurt_rank = rets.kurtosis().sort_values(ascending=False)
print('习题解答 2：超额峰度由大到小排名（QQ 图翘起幅度对应）')
print(kurt_rank.round(3))
print(f'\n厚尾最显著：{kurt_rank.index[0]}')

In [ ]:
# 习题解答 3：LIQUOR ACF + Ljung-Box
set_chinese_font()

col3 = 'LIQUOR'
r_liq = rets[col3].dropna()
NLAGS3 = 30
conf3 = 1.96 / np.sqrt(len(r_liq))

acf_r3   = sm_acf(r_liq.values, nlags=NLAGS3, fft=True)[1:]
acf_abs3 = sm_acf(r_liq.abs().values, nlags=NLAGS3, fft=True)[1:]
lags3 = np.arange(1, NLAGS3 + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'{col3} ACF 对比（习题 3）', fontsize=13)

for ax, acf_vals, title_sfx in [(ax1, acf_r3, '收益率'), (ax2, acf_abs3, '收益率绝对值')]:
    ax.bar(lags3, acf_vals, color='steelblue', alpha=0.8)
    ax.axhline(conf3, color='red', ls='--', lw=1)
    ax.axhline(-conf3, color='red', ls='--', lw=1)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{col3} {title_sfx} ACF')
    ax.set_xlabel('滞后阶数')
    ax.set_ylabel('自相关系数')

plt.tight_layout()
plt.show()

lb10_r   = acorr_ljungbox(r_liq, lags=[10], return_df=True)
lb10_abs = acorr_ljungbox(r_liq.abs(), lags=[10], return_df=True)
print(f'{col3} 收益率   Ljung-Box(10): Q={lb10_r.loc[10,"lb_stat"]:.2f}, p={lb10_r.loc[10,"lb_pvalue"]:.4f}')
print(f'{col3} |收益率| Ljung-Box(10): Q={lb10_abs.loc[10,"lb_stat"]:.2f}, p={lb10_abs.loc[10,"lb_pvalue"]:.4e}')
print('解释：收益率 p 值较大（不拒绝无自相关），|收益率| p 值极小（强烈拒绝，存在波动率聚集）')

In [ ]:
# 习题解答 5：TECH 不同频率分布对比
set_chinese_font()

col5 = 'TECH'
freq_data5 = {
    '日度': rets[col5],
    '周度': prices[[col5]].resample('W').last().pct_change().dropna()[col5],
    '月度': prices[[col5]].resample('ME').last().pct_change().dropna()[col5],
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle(f'{col5} 聚合高斯性：不同频率收益率分布（习题 5）', fontsize=13)

for ax, (label, data5) in zip(axes, freq_data5.items()):
    mu5, sigma5 = data5.mean(), data5.std()
    kurt5 = data5.kurtosis()
    ax.hist(data5, bins=40, density=True, alpha=0.4, color='steelblue')
    x_r5 = np.linspace(data5.min(), data5.max(), 300)
    ax.plot(x_r5, norm.pdf(x_r5, mu5, sigma5), 'r--', lw=2, label='正态拟合')
    kde5 = gaussian_kde(data5)
    ax.plot(x_r5, kde5(x_r5), 'b-', lw=1.5, label='KDE')
    ax.set_title(f'{label}（n={len(data5)}）\n超额峰度={kurt5:.3f}')
    ax.set_xlabel('收益率')
    ax.set_ylabel('概率密度')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('\n验证结论：')
for label, data5 in freq_data5.items():
    print(f'  {label}超额峰度 = {data5.kurtosis():.3f}')
print('→ 频率越低，超额峰度越小，分布越接近正态（聚合高斯性）')